# Week 4 Day 2 — Full Encoder Stack
**Jul 21, 2026**

Combine everything so far into one model that takes raw token IDs in and produces contextualized vectors out: an embedding table, Day 1's positional encoding, and `N` stacked copies of Week 3 Day 7's `EncoderBlock`. This is, quite literally, a working transformer encoder — the same architecture behind BERT.

Scaffold as usual: TODO stubs, hints not formulas, self-check cells.

## Part 1: The pieces, given

`MultiHeadAttention`, `EncoderBlock`, and `sinusoidal_positional_encoding` are given here exactly as built in Week 3 Day 7 / Week 4 Day 1 — copied in so this notebook is self-contained. Nothing new to implement in this cell; the new work starts in Part 2.

In [1]:
import torch
import torch.nn as nn
import math

torch.manual_seed(0)

def sinusoidal_positional_encoding(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    position = torch.arange(seq_len).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, S, D = x.shape
        Q = self.q_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim)
        weights = torch.softmax(scores, dim=-1)
        out = weights @ V
        out = out.transpose(1, 2).contiguous().view(B, S, D)
        return self.out_proj(out)

class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.norm1(x + self.attn(x))
        x = self.norm2(x + self.ff(x))
        return x

## Part 2: `TransformerEncoder`

TODO: build `TransformerEncoder(vocab_size, d_model, num_heads, d_ff, num_layers, max_len)`.

Pieces:
- `nn.Embedding(vocab_size, d_model)` — turns integer token IDs into `d_model`-dimensional vectors. This is the first time raw integers (not floats) are the model's input.
- A positional encoding table, precomputed once for `max_len` positions via `sinusoidal_positional_encoding`. Since it has no learnable parameters, register it with `self.register_buffer('pe', ...)` rather than as a plain attribute — a buffer moves with the module (e.g. `.to(device)`) and gets saved/loaded with `state_dict()` like a parameter would, but the optimizer correctly never touches it, since it's not something gradient descent should update.
- `num_layers` copies of `EncoderBlock`, held in an `nn.ModuleList` (the same registration reasoning from Week 3 Day 4 — a plain list would silently break training for every block).

In `forward(token_ids)`, where `token_ids` has shape `(batch, seq_len)`:
1. Embed the tokens, then scale the result by `sqrt(d_model)`. This detail is from the original paper — embeddings are initialized with roughly unit variance while the positional encoding's sin/cos values are bounded in `[-1, 1]` regardless of `d_model`; without this scaling, the positional signal can end up disproportionately large relative to the content signal once `d_model` grows, drowning out the embedding.
2. Add the positional encoding, sliced to the actual `seq_len` (the input might be shorter than `max_len`).
3. Run the result through each `EncoderBlock` in sequence — each block's output is the next block's input.

In [2]:
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_len=512):
        super().__init__()
        self.d_model = d_model
        # TODO: self.embedding, self.register_buffer('pe', ...), self.blocks
        self.embedding = nn.Embedding(vocab_size, d_model)
        pe = sinusoidal_positional_encoding(max_len, d_model)
        self.register_buffer('pe', pe)
        self.blocks = nn.ModuleList([EncoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)])
        

    def forward(self, token_ids):
        # TODO
        x = self.embedding(token_ids) * math.sqrt(self.d_model)
        seq_len = token_ids.size(1)
        x = x + self.pe[:seq_len, :].unsqueeze(0)
        for block in self.blocks:
            x = block(x)
        return x
       

vocab_size, d_model, num_heads, d_ff, num_layers = 50, 16, 2, 32, 3
encoder = TransformerEncoder(vocab_size, d_model, num_heads, d_ff, num_layers)
tokens = torch.randint(0, vocab_size, (2, 7))
out = encoder(tokens)

assert out.shape == (2, 7, d_model), f"expected (2, 7, {d_model}), got {tuple(out.shape)}"
print("shape OK")
print("total params:", sum(p.numel() for p in encoder.parameters()))

shape OK
total params: 7472


## Part 3: Check against `nn.TransformerEncoder`

Given, once Part 2 works. Same weight-copying oracle pattern as Week 3 Day 7, now looped across every layer in the stack. Note the embedding + positional encoding step is computed manually and fed directly into `ref_encoder`, since `nn.TransformerEncoder` itself doesn't include an embedding layer -- it only stacks the encoder layers.

In [3]:
encoder.eval()
with torch.no_grad():
    my_out = encoder(tokens)

ref_layer_factory = lambda: nn.TransformerEncoderLayer(
    d_model, num_heads, dim_feedforward=d_ff, batch_first=True, norm_first=False, activation="relu", dropout=0.0
)
ref_encoder = nn.TransformerEncoder(ref_layer_factory(), num_layers=num_layers)
ref_encoder.eval()

with torch.no_grad():
    for my_block, ref_layer in zip(encoder.blocks, ref_encoder.layers):
        ref_layer.self_attn.in_proj_weight.copy_(
            torch.cat([my_block.attn.q_proj.weight, my_block.attn.k_proj.weight, my_block.attn.v_proj.weight], dim=0)
        )
        ref_layer.self_attn.in_proj_bias.copy_(
            torch.cat([my_block.attn.q_proj.bias, my_block.attn.k_proj.bias, my_block.attn.v_proj.bias], dim=0)
        )
        ref_layer.self_attn.out_proj.weight.copy_(my_block.attn.out_proj.weight)
        ref_layer.self_attn.out_proj.bias.copy_(my_block.attn.out_proj.bias)
        ref_layer.linear1.weight.copy_(my_block.ff[0].weight); ref_layer.linear1.bias.copy_(my_block.ff[0].bias)
        ref_layer.linear2.weight.copy_(my_block.ff[2].weight); ref_layer.linear2.bias.copy_(my_block.ff[2].bias)
        ref_layer.norm1.weight.copy_(my_block.norm1.weight); ref_layer.norm1.bias.copy_(my_block.norm1.bias)
        ref_layer.norm2.weight.copy_(my_block.norm2.weight); ref_layer.norm2.bias.copy_(my_block.norm2.bias)

    embedded = encoder.embedding(tokens) * math.sqrt(d_model) + encoder.pe[:tokens.shape[1]]
    ref_out = ref_encoder(embedded)

assert torch.allclose(my_out, ref_out, atol=1e-4), "doesn't match nn.TransformerEncoder"
print("matches nn.TransformerEncoder -- full stack verified")

matches nn.TransformerEncoder -- full stack verified


## Try yourself

1. Remove the `sqrt(d_model)` scaling on the embeddings and rerun Part 2 -- shapes are unaffected, but think about (or empirically check, by comparing typical magnitudes) why the model's early training dynamics might differ.
2. Print `encoder.pe.requires_grad` -- confirm it's `False`, and explain why that's correct given how `register_buffer` differs from a plain parameter.
3. Try `num_layers=1` vs `num_layers=6` and compare total parameter counts -- is the relationship linear? Should it be, given every `EncoderBlock` has an identical structure?
4. Feed in a `token_ids` batch where sequences have different actual lengths (you'd normally pad shorter ones) -- nothing in this implementation handles padding specially. What would attending over padding tokens do to a real model's outputs, and what mechanism (hint: something you built in Week 3 Day 6's Try Yourself) could prevent the model from attending to padding positions?